# Visão geral das taxas de suicídio (1985–2016)

**Notebook principal — análise exploratória reproduzível**

Este notebook investiga diferenças descritivas nas taxas por sexo, faixa etária e geração. As taxas são calculadas a partir da razão entre o total de suicídios e o total da população em cada grupo, multiplicada por 100.000.

> Este material é educacional e observacional. Não permite inferir causalidade nem avaliar indivíduos.

## 1. Preparação do ambiente

Instale as dependências com `pip install -r requirements.txt` antes de executar.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="colorblind")
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path("../data/master.csv")


## 2. Carregamento e validação dos dados

O dataset deve ser salvo em `data/master.csv`. A validação abaixo evita que o notebook continue silenciosamente com um arquivo incompleto.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Arquivo não encontrado. Baixe o master.csv e salve-o em data/master.csv. "
        "Consulte data/README.md para obter instruções."
    )

df = pd.read_csv(DATA_PATH)
expected_columns = {
    "country", "year", "sex", "age", "suicides_no", "population",
    "suicides/100k pop", "generation"
}
missing_columns = expected_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")

print(f"Linhas: {len(df):,}")
print(f"Países: {df["country"].nunique()}")
print(f"Período: {df["year"].min()}–{df["year"].max()}")
df.head()


In [ ]:
missing = (df.isna().sum().sort_values(ascending=False).rename("nulos").to_frame())
missing["percentual"] = (missing["nulos"] / len(df) * 100).round(2)
missing[missing["nulos"] > 0]


## 3. Limpeza e definição do indicador

Em vez de somar taxas prontas, calculamos uma taxa agregada ponderada pela população: `soma dos suicídios / soma da população × 100.000`. Isso evita atribuir o mesmo peso a grupos populacionais de tamanhos diferentes.

In [ ]:
numeric_columns = ["year", "suicides_no", "population", "suicides/100k pop"]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.dropna(subset=["year", "sex", "age", "generation", "suicides_no", "population"]).copy()
df = df[df["population"] > 0].copy()

def aggregate_rate(data, dimensions):
    grouped = data.groupby(dimensions, as_index=False)[["suicides_no", "population"]].sum()
    grouped["rate_per_100k"] = grouped["suicides_no"] / grouped["population"] * 100_000
    return grouped


## 4. Taxa por sexo ao longo do tempo

In [ ]:
by_sex = aggregate_rate(df, ["year", "sex"])
plt.figure(figsize=(11, 6))
sns.lineplot(data=by_sex, x="year", y="rate_per_100k", hue="sex", marker="o")
plt.title("Taxa agregada por sexo ao longo do tempo")
plt.xlabel("Ano")
plt.ylabel("Taxa por 100 mil habitantes")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "taxa_por_sexo.png", dpi=160)
plt.show()


## 5. Taxa por faixa etária

In [ ]:
age_order = ["5-14 years", "15-24 years", "25-34 years", "35-54 years", "55-74 years", "75+ years"]
by_age = aggregate_rate(df, ["year", "age"])
by_age["age"] = pd.Categorical(by_age["age"], categories=age_order, ordered=True)
plt.figure(figsize=(11, 6))
sns.lineplot(data=by_age.sort_values("age"), x="year", y="rate_per_100k", hue="age")
plt.title("Taxa agregada por faixa etária")
plt.xlabel("Ano")
plt.ylabel("Taxa por 100 mil habitantes")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "taxa_por_faixa_etaria.png", dpi=160)
plt.show()


## 6. Taxa por geração

In [ ]:
by_generation = aggregate_rate(df, ["year", "generation"])
plt.figure(figsize=(11, 6))
sns.lineplot(data=by_generation, x="year", y="rate_per_100k", hue="generation")
plt.title("Taxa agregada por geração")
plt.xlabel("Ano")
plt.ylabel("Taxa por 100 mil habitantes")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "taxa_por_geracao.png", dpi=160)
plt.show()


## 7. Estatística descritiva e limitações

As estatísticas abaixo resumem as variáveis numéricas originais. A transformação `log1p`, quando necessária, deve ser aplicada a uma cópia da variável e nunca substituir o indicador original sem documentação, especialmente porque existem valores iguais a zero.

In [ ]:
summary = df[["year", "suicides_no", "population", "suicides/100k pop"]].describe().T
summary["mediana"] = df[["year", "suicides_no", "population", "suicides/100k pop"]].median()
summary


## 8. Conclusões responsáveis

As visualizações permitem comparar padrões descritivos por sexo, idade e geração, mas não demonstram que qualquer uma dessas características cause o desfecho. Diferenças observadas podem refletir composição populacional, cobertura desigual, subnotificação, mudanças de registro e outros fatores não medidos.

Para uma investigação mais robusta, seria necessário complementar a fonte com bases oficiais, documentar a cobertura por país, controlar possíveis confundidores e aplicar métodos estatísticos apropriados ao desenho dos dados.